In [11]:
uv add pandas

/Users/oscaralawi/code/MUlbing/ClaimLens/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install pandas

/Users/oscaralawi/code/MUlbing/ClaimLens/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import json
import pandas as pd

print("Ready")

Ready


In [15]:
from pathlib import Path

path = Path("abcd_v1.1.json")
if not path.exists():
    path = Path("data/raw/abcd_v1.1.json")

with path.open(encoding="utf-8") as f:
    data = json.load(f)

for split, conversations in data.items():
    print(split, len(conversations))

train 8034
dev 1004
test 1004


In [16]:
example = data["train"][0]

print("Fields:", list(example.keys()))
print("Conversation ID:", example["convo_id"])
print("Scenario:", example["scenario"])

Fields: ['convo_id', 'scenario', 'original', 'delexed']
Conversation ID: 3592
Scenario: {'personal': {'customer_name': 'crystal minh', 'email': 'cminh730@email.com', 'member_level': 'bronze', 'phone': '(977) 625-2661', 'username': 'cminh730'}, 'order': {'street_address': '6821 1st ave', 'full_address': '6821 1st ave  san mateo, ny 75227', 'city': 'san mateo', 'num_products': '1', 'order_id': '3348917502', 'packaging': 'yes', 'payment_method': 'credit card', 'products': "[{'brand': 'michael_kors', 'product_type': 'jeans', 'amount': 94, 'image_url': 'images/michael_kors-jeans.jpeg'}]", 'purchase_date': '2019-11-06', 'state': 'ny', 'zip_code': '75227'}, 'product': {'names': ['michael_kors jeans'], 'amounts': [94]}, 'flow': 'product_defect', 'subflow': 'return_size'}


In [17]:
for entry in example["original"][:5]:
    print(entry)

['agent', 'Hi!']
['agent', 'How can I help you?']
['customer', 'Hi! I need to return an item, can you help me with that?']
['agent', 'sure, may I have your name please?']
['customer', 'Crystal Minh']


In [20]:
rows = []

for split, conversations in data.items():
    for convo in conversations:
        rows.append({
            "convo_id": convo["convo_id"],
            "split": split,
            "flow": convo["scenario"]["flow"],
            "subflow": convo["scenario"]["subflow"],
            "customer_text": " ".join(
                text for speaker, text in convo["original"]
                if speaker == "customer"
            )
        })

df = pd.DataFrame(rows)
display(df.head())
print("Table shape:", df.shape)

,convo_id,split,flow,subflow,customer_text
0,3592,train,product_defect,return_size,"Hi! I need to return an item, can you help me ..."
1,9489,train,product_defect,refund_status,just wanted to check on the status of a refund...
2,3695,train,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...
3,5798,train,shipping_issue,manage,"Hello, I would like to change my shipping deat..."
4,3647,train,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...


Table shape: (10042, 5)


In [21]:
print("Missing values per column:")
print(df.isna().sum())

print("\nDuplicate conversation IDs:")
print(df["convo_id"].duplicated().sum())

print("\nEmpty customer texts:")
print(df["customer_text"].str.strip().eq("").sum())

Missing values per column:
convo_id         0
split            0
flow             0
subflow          0
customer_text    0
dtype: int64

Duplicate conversation IDs:
0

Empty customer texts:
0


In [22]:
print("Unique flows:", df["flow"].nunique())
print("Unique subflows:", df["subflow"].nunique())

train_df = df[df["split"] == "train"].copy()

print("\nTraining conversations per flow:")
display(train_df["flow"].value_counts())

Unique flows: 10
Unique subflows: 96

Training conversations per flow:


flow
storewide_query         872
product_defect          863
purchase_dispute        858
account_access          847
single_item_query       840
order_issue             831
troubleshoot_site       819
shipping_issue          814
subscription_inquiry    718
manage_account          572
Name: count, dtype: int64

In [23]:
pattern = r"\brefund\w*\b|\bmoney\s+back\b"

train_df["refund_mention"] = train_df["customer_text"].str.contains(
    pattern, case=False, na=False
)

refund_candidates = train_df[train_df["refund_mention"]].copy()

print("Conversations mentioning refunds:", len(refund_candidates))
display(refund_candidates[["convo_id", "subflow", "customer_text"]].head())

Conversations mentioning refunds: 702


,convo_id,subflow,customer_text
1,9489,refund_status,just wanted to check on the status of a refund...
16,1699,mistimed_billing_already_returned,"Hi, i had a return but you guys never refunded..."
28,1657,refund_update,I need help with a refund Allessandro Phoenix ...
29,2604,policy_2,"hi, I'm thinking of buying something but I'm c..."
34,991,refund_status,"Hi. I am checking on my refund? Crystal Minh, ..."


In [24]:
refund_summary = (
    train_df.groupby(["flow", "subflow"])["refund_mention"]
    .agg(total_conversations="size", refund_mentions="sum")
    .reset_index()
)

refund_summary["mention_pct"] = (
    100 * refund_summary["refund_mentions"]
    / refund_summary["total_conversations"]
).round(1)

refund_summary = refund_summary.sort_values(
    "refund_mentions", ascending=False
)

display(refund_summary.head(20))

,flow,subflow,total_conversations,refund_mentions,mention_pct
21,product_defect,refund_status,145,141,97.2
22,product_defect,refund_update,146,140,95.9
20,product_defect,refund_initiate,139,128,92.1
34,shipping_issue,cost,194,84,43.3
75,storewide_query,policy_2,54,38,70.4
8,manage_account,status_service_added,104,37,35.6
86,subscription_inquiry,manage_dispute_bill,145,29,20.0
28,purchase_dispute,mistimed_billing_already_returned,114,25,21.9
7,manage_account,status_credit_missing,90,14,15.6
29,purchase_dispute,mistimed_billing_never_bought,116,9,7.8


In [25]:
sample_id = refund_candidates.loc[
    (refund_candidates["flow"] == "shipping_issue")
    & (refund_candidates["subflow"] == "cost"),
    "convo_id"
].iloc[0]

conversation = next(
    convo for convo in data["train"]
    if convo["convo_id"] == sample_id
)

print("Conversation ID:", sample_id)
for speaker, text in conversation["original"]:
    print(f"{speaker}: {text}")

Conversation ID: 2461
agent: Hello, how can I help you today?
customer: HI. I want to check on my order status. I paid a lot for shipping for the item and haven't received it.
agent: Happy to help, may I have your full name please?
customer: Chloe Zhang
action: Account has been pulled up for Chloe Zhang.
agent: thanks Chloe.
customer: Is it possible to cancel the order and get a refund?
agent: Got it. To verify your identity, I also need your account id and order id please.
customer: ZB880QPBJZ is my account id
customer: My order ID is 2663462033
agent: thanks Chloe.
action: Identity verification in progess ...
agent: Your identity is verified. What is the shipping status of your order?
customer: Out for Delivery
agent: Your membership level may get some benefits in this situation. What is your membership level?
customer: Gold
action: Membership level of Gold has been noted.
agent: I can offer a refund immediately.
agent: After the item arrives, you should then call back to customer su

In [26]:
conversation_lookup = {
    convo["convo_id"]: "\n".join(
        f"{speaker}: {text}"
        for speaker, text in convo["original"]
    )
    for convo in data["train"]
}

refund_candidates["conversation_text"] = (
    refund_candidates["convo_id"].map(conversation_lookup)
)

print(refund_candidates["conversation_text"].iloc[0])

agent: good afternoon, how can I help you?
customer: just wanted to check on the status of a refund
agent: sure, would you give me your full name or account ID
customer: Alessandro Phoenix
customer: aphoenix939
action: Account has been pulled up for Alessandro Phoenix.
agent: additional to this you would give me the order ID and email
agent: please
customer: 7916676427
customer: aphoenix939@email.com
customer: no worries
action: Purchase validation in progress ...
agent: everything in order, soon I will indicate the status of your refund.
customer: great
customer: let me know
agent: It is currently in progress and the payment method with which it is being processed is online towards your credit card.
customer: how much long till it is refunded
agent: less than a week
customer: great thanks for your help
agent: a pleasure to help you
agent: have a nice day


In [27]:
sample = refund_candidates.sample(1, random_state=42).iloc[0]

print("Conversation ID:", sample["convo_id"])
print("Issue:", sample["flow"], "/", sample["subflow"])
print()
print(sample["conversation_text"])

Conversation ID: 1251
Issue: product_defect / refund_update

customer: hiya
agent: Hello, thank you for contacting us. How can I help you?
customer: so i have a refund that's already in progress, however, i need to add one more item to it
agent: That shouldn't be a problem. Can I have your first and last name please?
customer: sure
customer: it's Sanya Afzal
action: Account has been pulled up for Sanya Afzal.
agent: Alright, So we need to cancel and item and add it to your refund correct?
customer: right, my existing refund is $100, so i'd just need the other item added (the $79 michael kors jeans)
agent: Awesome, thank you for the additional information. Can I please have your Account ID and Order ID?
customer: sure
customer: Order ID: 2600248569
customer: i don't have an account id available, sorry
agent: Alright give me one moment to see what I can do to help.
customer: thanks
agent: Sorry for the delay, I'm inquiring with my manager what to do without an account ID.
customer: alrig

In [28]:
conversation = next(
    convo for convo in data["train"]
    if convo["convo_id"] == sample["convo_id"]
)

keywords = ["refund", "added", "$79", "$100"]

for speaker, text in conversation["original"]:
    if speaker in ["customer", "agent"]:
        if any(word in text.lower() for word in keywords):
            print(f"{speaker}: {text}\n")

customer: so i have a refund that's already in progress, however, i need to add one more item to it

agent: Alright, So we need to cancel and item and add it to your refund correct?

customer: right, my existing refund is $100, so i'd just need the other item added (the $79 michael kors jeans)

agent: Alright and the refund was for 79$ for this item?

agent: Alright the item was successfully added to the refund. Was there anything else I could help you with?



In [29]:
missed_candidates = train_df[
    (train_df["subflow"] == "refund_initiate")
    & (~train_df["refund_mention"])
]

print("Conversations to inspect:", len(missed_candidates))

for row in missed_candidates.head(3).itertuples():
    print(f"\nConversation {row.convo_id}")
    print(row.customer_text)

Conversations to inspect: 11

Conversation 9081
Hi I am needing to return an item I just purchased because I changed my mind. Norman Bouchard Nothing wrong with it Great thank you! Username: normanb8 Email Address: normanb8@email.com Order ID: 5369992383 credit card please Account ID: W4HRDGFEEX $84 Thank you. That's it for today.

Conversation 8065
Hi I have a question about your return policy I bought a pair of boots for a birthday present but it's not the right size. I wanted to know how I can return it? hmmm, my status is bronze I made the purchase in September of 2019 I think I have the receipt somewhere ok cool that's great how can I get the return started? ok can I get that started now? I want to start a return but I wanted to know how the policy worked Joyce Wu jwu55057 jwu55057@email.com 1813099703 ok cool yes 5555 Lincoln Ave  Newark, MI 46324 mail great thanks so much! that's it

Conversation 9435
Hi, my name's Albert Sanders I just ordered a jacket, but I'm afraid I've chan

In [30]:
def get_conversation(convo_id, dataset):
    for split, conversations in dataset.items():
        for convo in conversations:
            if convo["convo_id"] == convo_id:
                turns = pd.DataFrame(
                    convo["original"],
                    columns=["speaker", "text"]
                )
                turns.insert(0, "turn", range(len(turns)))
                return turns

    raise ValueError(f"Conversation {convo_id} not found")

In [32]:
chat = get_conversation(9081, data)

with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(chat)

,turn,speaker,text
0,0,agent,Thank you for contacting Acme Brands! How can I help you today?
1,1,customer,Hi I am needing to return an item I just purchased because I changed my mind.
2,2,agent,I can help you with that. Could I have your name or account ID to pull up your account?
3,3,customer,Norman Bouchard
4,4,action,Account has been pulled up for Norman Bouchard.
5,5,agent,Thank you. There wasn't anything wrong with the product?
6,6,customer,Nothing wrong with it
7,7,agent,"Alright. I should be able to provide you with a refund, if you just placed the order and it hasn't shipped."
8,8,agent,"Can you verify your username, email address and the order ID for me?"
9,9,customer,Great thank you!


In [33]:
audit_case = pd.DataFrame([{
    "convo_id": 9081,
    "customer_intent": "return_changed_mind",
    "refund_outcome": "issued",
    "refund_amount": 84,
    "evidence_turns": "14, 22, 23",
    "keyword_search_missed": True
}])

display(audit_case)

,convo_id,customer_intent,refund_outcome,refund_amount,evidence_turns,keyword_search_missed
0,9081,return_changed_mind,issued,84,"14, 22, 23",True


In [34]:
def build_conversation_table(dataset):
    rows = []

    for split, conversations in dataset.items():
        for convo in conversations:
            messages = convo["original"]
            rows.append({
                "convo_id": convo["convo_id"],
                "split": split,
                "flow": convo["scenario"]["flow"],
                "subflow": convo["scenario"]["subflow"],
                "customer_text": "\n".join(
                    text for speaker, text in messages
                    if speaker == "customer"
                ),
                "agent_text": "\n".join(
                    text for speaker, text in messages
                    if speaker == "agent"
                ),
                "action_text": "\n".join(
                    text for speaker, text in messages
                    if speaker == "action"
                )
            })

    table = pd.DataFrame(rows)
    table["refund_mention"] = table["customer_text"].str.contains(
        r"\brefund\w*\b|\bmoney\s+back\b",
        case=False, na=False
    )
    return table

In [35]:
conversation_table = build_conversation_table(data)

display(conversation_table.head())
print("Table shape:", conversation_table.shape)

,convo_id,split,flow,subflow,customer_text,agent_text,action_text,refund_mention
0,3592,train,product_defect,return_size,"Hi! I need to return an item, can you help me ...","Hi!\nHow can I help you?\nsure, may I have you...",Account has been pulled up for Crystal Minh.\n...,False
1,9489,train,product_defect,refund_status,just wanted to check on the status of a refund...,"good afternoon, how can I help you?\nsure, wou...",Account has been pulled up for Alessandro Phoe...,True
2,3695,train,storewide_query,timing_4,HEY HO!\nI've got a promo code and I want to k...,"good afternoon, how can I help you?\nsure! le...",Searching the FAQ pages ...\nSystem Action: se...,False
3,5798,train,shipping_issue,manage,"Hello, I would like to change my shipping deat...",Welcome to AcmeBrands! How can I help you?\nI ...,Account has been pulled up for Crystal Minh.\n...,False
4,3647,train,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,"Hello, how can i help you\nYou want to know ho...",Searching the FAQ pages ...\nSystem Action: se...,False


Table shape: (10042, 8)


In [36]:
conversation_table["recorded_refund_amount"] = pd.to_numeric(
    conversation_table["action_text"].str.extract(
        r"A refund has been made for the amount of \$([\d,]+(?:\.\d+)?)",
        expand=False
    ).str.replace(",", "", regex=False),
    errors="coerce"
)

display(conversation_table.loc[
    conversation_table["convo_id"] == 9081,
    ["convo_id", "refund_mention", "recorded_refund_amount"]
])

,convo_id,refund_mention,recorded_refund_amount
617,9081,False,84.0


In [37]:
def prepare_model_data(dataset, labels):
    rows = []

    for split, conversations in dataset.items():
        for convo in conversations:
            first_message = next(
                (text for speaker, text in convo["original"]
                 if speaker == "customer"),
                ""
            )
            rows.append({
                "convo_id": convo["convo_id"],
                "split": split,
                "text": first_message
            })

    table = pd.DataFrame(rows).merge(
        labels[["convo_id", "decision"]],
        on="convo_id",
        how="inner",
        validate="one_to_one"
    )
    table = table[table["decision"].isin(["accepted", "rejected"])]
    table["target"] = table["decision"].map({
        "accepted": 1, "rejected": 0
    })
    return table